# Targeted fine-tuning — Leeds parking

**What this asks.** The first fine-tuning run fed the model every Leeds label and let it work
things out. It raised IoU by 0.128 but bought precision with recall, and because the model
contracted everywhere, the fall in standalone false positives cannot be attributed to any
particular confusion class. This notebook closes that loop: it uses the Chapter 4 error
typology to build the supervision, then measures whether the removal is *selective*.

**Design.**

1. Run the released zero-shot model on the **training half only** and find its own FP and FN.
2. Attribute the **standalone** FP (> 5 m from any label) to the same reference layers used in
   §4.2 — road buffers, building curtilage, OSM parking, sports, brownfield, industrial.
3. Turn that into a per-pixel loss weight map: hard negatives on the attributed FP, extra
   weight on FN, **weight 1 on boundary FP** so the model is not taught to shrink.
4. Fine-tune with weighted cross-entropy. Everything else — optimiser, LR, batch, epochs,
   seed, epoch-selection rule — is identical to the generic run, so the loss weighting is the
   only difference.
5. Evaluate **three arms** on the same 50 held-out cells with FP broken down by category, and
   report the removal rate per category for each arm.

**The result this exists to produce** is the last table: if targeting worked, the targeted
model removes more of the definitional categories than the generic model does, and keeps more
recall. If the removal rates are flat across categories in both arms, then both models simply
contracted, and §4.8 must say so.

**Nothing in `fine-tuning/` is modified.** `modeling.py` and `patch_data.py` are imported from
it read-only so the preprocessing contract cannot drift. All outputs go to a separate Drive
folder.

**Runtime** roughly 1.5–2.5 h end to end on a T4; faster on A100. Arms A and C always run;
arm B (the generic model) runs only if the first experiment's `finetuned.ckpt` is in Drive.

## 1. Configuration, Drive and GPU

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, shutil, subprocess, json, math

REPO_URL   = 'https://github.com/hou1020/Parking.git'
BRANCH     = 'main'
REPO       = Path('/content/Parking')
FT         = REPO / 'fine-tuning'            # read-only reuse
TG         = REPO / 'targeted-finetuning'

# The first experiment's Drive folder is READ ONLY here: we borrow its checkpoint and its
# prepared patches.  Everything this notebook produces goes to its own folder.
GENERIC_RUN = Path('/content/drive/MyDrive/Parking_finetuning_run')
RUN         = Path('/content/drive/MyDrive/Parking_targeted_run')
CACHE       = RUN / 'cache'
RUN.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

# --- identical to the generic run, so the comparison is controlled -------------------
EPOCHS, TRAIN_BATCH, EVAL_BATCH, LR, SEED, NUM_WORKERS = 6, 2, 4, 2e-5, 42, 2

# --- the only thing that differs: typology-driven per-pixel loss weights -------------
# Codes written by section 7:
#   0 ordinary pixel (includes boundary/dilation FP - deliberately NOT upweighted)
#   1 standalone FP that no layer explains
#   2 false negative (missed parking)
#   3 standalone FP on a precise layer   (road / curtilage / OSM parking / sports)
#   4 standalone FP on a broad land-use layer (brownfield / industrial)
W_ORDINARY, W_FP_OTHER, W_FN, W_FP_PRECISE, W_FP_BROAD = 1.0, 1.0, 3.0, 5.0, 2.0

DILATION_M   = 5.0    # same working threshold as Chapter 4
EXTRA_ROAD_M = 6.0    # same road widening as fp_analysis.py
CURTILAGE_M  = 8.0    # buildings buffered outward: private forecourt / driveway proxy
PIXEL_M      = 0.25
CELL_PX      = 4000

FORCE_REBUILD_LAYERS = False
FORCE_REBUILD_CODES  = False
FORCE_RETRAIN        = False

# NOTE: torch / numpy are deliberately NOT imported here.  Section 2 installs packages, and
# anything already loaded into this process would keep its old C extensions while pip
# replaces the files on disk.  That mismatch is what produces
#     AttributeError: module 'numpy._core._multiarray_umath' has no attribute ...
print('run folder:', RUN)

Mounted at /content/drive
run folder: /content/drive/MyDrive/Parking_targeted_run


## 2. Dependencies

`transformers` is pinned to the version the released checkpoint's key names belong to.

Two rules keep this cell from breaking the runtime:

* **no `--upgrade`** — Colab already ships working numpy, scipy, pandas, Pillow and torch, and
  upgrading them churns numpy for no benefit;
* **install before importing** — and if numpy or scipy did change anyway (a dependency of
  geopandas or rasterio can force it), the cell restarts the runtime itself. Colab will
  reconnect; just run all cells again from the top and this cell will be a no-op.

In [3]:
import importlib.metadata as md

def ver(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None

if shutil.which('git-lfs') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git-lfs'], check=True)

if shutil.which('git-lfs') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'git-lfs'], check=True)

todo = []
if ver('transformers') != '4.57.1':
    todo.append('transformers==4.57.1')
for pkg in ['geopandas', 'rasterio', 'tifffile', 'huggingface_hub', 'shapely']:
    if ver(pkg) is None:
        todo.append(pkg)

if todo:
    before = (ver('numpy'), ver('scipy'))
    print('installing:', ' '.join(todo))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *todo], check=True)
    after = (ver('numpy'), ver('scipy'))
    if before != after:
        print(f'numpy/scipy changed {before} -> {after}')
        print('RESTARTING THE RUNTIME. When it reconnects, Run all from the top.')
        os.kill(os.getpid(), 9)
else:
    print('all dependencies already satisfied')

assert ver('transformers') == '4.57.1', ver('transformers')
print('transformers', ver('transformers'), '| geopandas', ver('geopandas'),
      '| rasterio', ver('rasterio'), '| numpy', ver('numpy'), '| scipy', ver('scipy'))

# Safe to import heavy libraries only now that the environment is settled.
import torch
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))

installing: transformers==4.57.1
transformers 4.57.1 | geopandas 1.1.4 | rasterio 1.5.0 | numpy 2.0.2 | scipy 1.16.3
GPU: Tesla T4


## 3. Clone the repository

LFS smudging is off during clone; only the 100 source TIFFs are pulled. The reference layers
(`analysis/ref_cache.gpkg`, `analysis/osm_extra.gpkg`, the OS Greenspace shapefile and the 100
`osm_cache` road/building tiles) are ordinary tracked files and arrive with the clone.

In [4]:
env = os.environ.copy(); env['GIT_LFS_SKIP_SMUDGE'] = '1'
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO)],
                   env=env, check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)

subprocess.run(['git', '-C', str(REPO), 'lfs', 'install', '--local'], check=True)
subprocess.run(['git', '-C', str(REPO), 'lfs', 'pull',
                '--include=parking-lot-mapping-tool/files/tif/**'], check=True)

TIF_ROOT = REPO / 'parking-lot-mapping-tool' / 'files' / 'tif'
n_tif = len(list(TIF_ROOT.rglob('*.tif')))
print('source TIFFs:', n_tif)
assert n_tif >= 100, 'LFS pull incomplete'

for p in ['analysis/ref_cache.gpkg', 'analysis/osm_extra.gpkg',
          'fine-tuning/leeds_manual.gpkg', 'fine-tuning/leeds_grid.gpkg', 'fine-tuning/split.csv']:
    assert (REPO / p).exists(), f'missing {p}'
print('reference layers present')

sys.path.insert(0, str(FT))          # modeling.py / patch_data.py, imported read-only
TG.mkdir(parents=True, exist_ok=True)

source TIFFs: 100
reference layers present


## 4. Prepared patches

Reuses the masks and `patch_index.csv` built by the first experiment. If the Drive cache from
that run is present it is restored; otherwise `make_patches.py` rebuilds them, which is
deterministic and produces the identical 2,256 / 3,200 split.

In [5]:
def prepared_ok(root):
    root = Path(root)
    return ((root / 'patch_index.csv').exists()
            and len(list((root / 'patches/train/Masks').glob('*.png'))) == 2256
            and len(list((root / 'patches/test/Masks').glob('*.png'))) == 3200)

GENERIC_CACHE = GENERIC_RUN / 'prepared_data'
if prepared_ok(FT):
    print('prepared data already in runtime')
elif prepared_ok(GENERIC_CACHE):
    shutil.copy2(GENERIC_CACHE / 'patch_index.csv', FT / 'patch_index.csv')
    shutil.copytree(GENERIC_CACHE / 'patches', FT / 'patches', dirs_exist_ok=True)
    print('restored prepared data from the first run\'s Drive cache')
else:
    print('building patches from the committed GPKGs (a few minutes) ...')
    subprocess.run([sys.executable, 'make_patches.py'], cwd=str(FT), check=True)

assert prepared_ok(FT), 'patch preparation failed'

import pandas as pd, numpy as np
IDX   = pd.read_csv(FT / 'patch_index.csv')
SPLIT = pd.read_csv(FT / 'split.csv')
print(IDX.groupby('split').size().to_string())
print('cells:', SPLIT.groupby('split').size().to_dict())

restored prepared data from the first run's Drive cache
split
test     3200
train    3200
cells: {'test': 50, 'train': 50}


## 5. Checkpoints

Arm A is the released model. Arm B is the first experiment's generic fine-tune — optional; if its checkpoint is not in Drive the notebook runs A and C and says so.

In [6]:
from huggingface_hub import hf_hub_download

ZERO_SHOT = GENERIC_RUN / 'best_model.ckpt'
if not ZERO_SHOT.exists() or ZERO_SHOT.stat().st_size < 1_000_000_000:
    ZERO_SHOT = RUN / 'best_model.ckpt'
    if not ZERO_SHOT.exists() or ZERO_SHOT.stat().st_size < 1_000_000_000:
        hf_hub_download(repo_id='UTEL-UIUC/SegFormer-large-parking',
                        filename='best_model.ckpt', local_dir=str(RUN))
assert ZERO_SHOT.stat().st_size > 1_000_000_000
print(f'arm A  zero-shot     : {ZERO_SHOT}  ({ZERO_SHOT.stat().st_size/1e9:.3f} GB)')

GENERIC_CKPT = GENERIC_RUN / 'finetuned.ckpt'
HAVE_GENERIC = GENERIC_CKPT.exists() and GENERIC_CKPT.stat().st_size > 1_000_000
print('arm B  generic FT    :', GENERIC_CKPT if HAVE_GENERIC else 'NOT FOUND - arm B skipped')

TARGETED_CKPT = RUN / 'targeted.ckpt'
print('arm C  targeted FT   :', TARGETED_CKPT, '(this notebook)')

arm A  zero-shot     : /content/drive/MyDrive/Parking_finetuning_run/best_model.ckpt  (1.017 GB)
arm B  generic FT    : /content/drive/MyDrive/Parking_finetuning_run/finetuned.ckpt
arm C  targeted FT   : /content/drive/MyDrive/Parking_targeted_run/targeted.ckpt (this notebook)


## 6. Category rasters

The six layers of §4.2 are rasterised onto each cell's own 4,000 × 4,000 grid and packed as
bitplanes in one `uint8` array per cell. `curtilage` is new here: buildings buffered outward by
8 m and the footprints removed, standing in for the private forecourts and driveways that the
sampling estimated at 20.2% of the unexplained residual. It is a proxy, and §4.2's caveat
applies to it more than to any other layer — attribution is by location, not by inspection.

In [7]:
import geopandas as gpd
from shapely.ops import unary_union
from shapely.geometry import box
from rasterio.features import rasterize
from rasterio.transform import from_origin
import warnings; warnings.filterwarnings('ignore')

LAYERS = ['building', 'osm_parking', 'sports', 'road_wide', 'curtilage', 'brownfield', 'industrial']
BIT    = {n: i for i, n in enumerate(LAYERS)}
PRECISE = ['osm_parking', 'sports', 'road_wide', 'curtilage']     # -> code 3
BROAD   = ['brownfield', 'industrial']                            # -> code 4
# Peeling order for attribution: most specific evidence first, exactly as fp_analysis.py,
# with curtilage inserted next to road_adjacent because both are proximity rules.
PEEL = ['building', 'osm_parking', 'sports', 'road_wide', 'curtilage', 'brownfield', 'industrial']

LAYER_NPZ = CACHE / 'category_layers.npz'


def _dissolve(gdf):
    g = gdf.to_crs(27700).copy()
    g = g[g.geometry.type.isin(['Polygon', 'MultiPolygon'])]
    if not len(g):
        return None
    g['geometry'] = g.geometry.buffer(0)
    return unary_union(g.geometry.values)


def build_layer_geometries():
    print('loading reference layers ...')
    ref = gpd.read_file(REPO / 'analysis/ref_cache.gpkg').to_crs(27700)
    buildings = _dissolve(ref[ref['grp'] == 'buildings'])
    roads     = _dissolve(ref[ref['grp'] == 'roads'])

    ex  = gpd.read_file(REPO / 'analysis/osm_extra.gpkg').to_crs(27700)
    grp = lambda k: _dissolve(ex[ex['grp'] == k]) if (ex['grp'] == k).any() else None

    gs = gpd.read_file(REPO / 'analysis/OS Open Greenspace (ESRI Shape File) SE/data/SE_GreenspaceSite.shp').to_crs(27700)
    gs_sports = _dissolve(gs[gs['function'].isin(['Tennis Court', 'Other Sports Facility', 'Play Space'])])

    pitch  = grp('pitch')
    sports = unary_union([g for g in (gs_sports, pitch) if g is not None])

    geoms = {
        'building':    buildings,
        'osm_parking': grp('osm_parking'),
        'sports':      sports,
        'road_wide':   roads.buffer(EXTRA_ROAD_M) if roads is not None else None,
        'curtilage':   buildings.buffer(CURTILAGE_M).difference(buildings) if buildings is not None else None,
        'brownfield':  grp('brownfield_bare'),
        'industrial':  grp('industrial_yard'),
    }
    for k, v in geoms.items():
        print(f'  {k:<12} {"ok" if v is not None and not v.is_empty else "EMPTY"}')
    return geoms


def build_category_rasters():
    geoms = build_layer_geometries()
    parts = {k: gpd.GeoSeries([g], crs=27700).explode(index_parts=False).reset_index(drop=True)
             for k, g in geoms.items() if g is not None and not g.is_empty}
    sidx = {k: v.sindex for k, v in parts.items()}

    out = {}
    for n, row in enumerate(SPLIT.itertuples(), 1):
        tf  = from_origin(row.left, row.top, PIXEL_M, PIXEL_M)
        bb  = box(row.left, row.bottom, row.right, row.top)
        packed = np.zeros((CELL_PX, CELL_PX), dtype=np.uint8)
        for name, gs_parts in parts.items():
            hit = sidx[name].query(bb, predicate='intersects')
            if len(hit) == 0:
                continue
            shapes = [(g, 1) for g in gs_parts.iloc[hit].values if g is not None and not g.is_empty]
            if not shapes:
                continue
            r = rasterize(shapes, out_shape=(CELL_PX, CELL_PX), transform=tf, fill=0,
                          dtype='uint8', all_touched=False)
            packed |= (r.astype(np.uint8) << BIT[name])
        out[row.cell] = packed
        if n % 20 == 0 or n == len(SPLIT):
            print(f'  rasterised {n}/{len(SPLIT)} cells')
    return out


if LAYER_NPZ.exists() and not FORCE_REBUILD_LAYERS:
    print('using cached category rasters')
else:
    _built = build_category_rasters()
    print('saving cache to Drive ...')
    np.savez_compressed(LAYER_NPZ, **_built)
    del _built

# Kept lazy on purpose: holding all 100 cells would pin ~1.6 GB of RAM, and each cell is
# touched only twice in the whole notebook.  NpzFile decompresses on access.
CATEGORY = np.load(LAYER_NPZ)
assert len(CATEGORY.files) == 100, len(CATEGORY.files)
_c = CATEGORY[CATEGORY.files[0]]
print('cells:', len(CATEGORY.files), '| shape', _c.shape, '| coverage % in cell 0:',
      {n: round(100 * float(((_c >> BIT[n]) & 1).mean()), 1) for n in LAYERS})
del _c

using cached category rasters
cells: 100 | shape (4000, 4000) | coverage % in cell 0: {'building': 8.4, 'osm_parking': 1.0, 'sports': 0.2, 'road_wide': 17.8, 'curtilage': 18.4, 'brownfield': 13.7, 'industrial': 9.8}


## 7. Zero-shot error maps → per-pixel weight codes

Run the released model over the **training half only** and turn its own mistakes into
supervision. Two choices carry the design:

* **Boundary FP is left at weight 1.** Upweighting FP that lies within 5 m of a real car park
  is precisely how a model is taught to draw everything smaller — the failure mode of the
  generic run. Only *standalone* FP becomes a hard negative.
* **FN is upweighted.** The generic run lost 0.127 of recall; weighting missed parking pushes
  the other way, so the two effects can be separated in the evaluation.

The held-out 50 cells are never read here.

In [8]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from PIL import Image
from scipy import ndimage
from modeling import make_processor, make_model, load_checkpoint
from patch_data import SourcePatchDataset, TileBatchSampler

CODE_ROOT = FT / 'patches' / 'train' / 'Codes'      # generated data, gitignored area
device = torch.device('cuda')
DIL_PX = DILATION_M / PIXEL_M


def assemble(pairs):
    """Rebuild a full 4000x4000 boolean cell from its 64 (row, array) pairs."""
    cell = np.zeros((CELL_PX, CELL_PX), dtype=bool)
    for r, arr in pairs:
        vh, vw = r['valid_h'], r['valid_w']
        cell[r['row_off']:r['row_off'] + vh, r['col_off']:r['col_off'] + vw] = arr[:vh, :vw]
    return cell


@torch.no_grad()
def predict_rows(model, rows, patch_root, batch_size=EVAL_BATCH):
    ds = SourcePatchDataset(rows, patch_root, str(TIF_ROOT), make_processor(), return_index=True)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
                    pin_memory=True)
    preds = [None] * len(rows)
    for batch, ids in dl:
        logits = model(pixel_values=batch['pixel_values'].to(device, non_blocking=True))[0]
        up = F.interpolate(logits, size=batch['labels'].shape[-2:], mode='bilinear',
                           align_corners=False)
        p = up.argmax(1).cpu().numpy().astype(bool)
        for b, i in enumerate(ids.tolist()):
            preds[i] = p[b]
    return preds


def build_weight_codes():
    CODE_ROOT.mkdir(parents=True, exist_ok=True)
    model = load_checkpoint(make_model(), str(ZERO_SHOT)).to(device).eval()
    train_rows = IDX[(IDX['split'] == 'train') & IDX['kept']].sort_values(
        ['cell', 'row_off', 'col_off']).to_dict('records')

    stats = {c: 0 for c in range(5)}
    cells = sorted({r['cell'] for r in train_rows})
    for n, cell in enumerate(cells, 1):
        rows = [r for r in train_rows if r['cell'] == cell]
        preds = predict_rows(model, rows, str(FT / 'patches' / 'train'))

        masks = []
        for r in rows:
            with Image.open(FT / 'patches' / 'train' / 'Masks' / r['name']) as im:
                masks.append(np.array(im) == 1)
        ref  = assemble(list(zip(rows, masks)))
        pred = assemble(list(zip(rows, preds)))

        fp, fn = pred & ~ref, ~pred & ref
        if ref.any():
            standalone = fp & (ndimage.distance_transform_edt(~ref) > DIL_PX)
        else:
            standalone = fp

        bits = CATEGORY[cell]
        precise = np.zeros_like(ref); broad = np.zeros_like(ref)
        for name in PRECISE:
            precise |= ((bits >> BIT[name]) & 1).astype(bool)
        for name in BROAD:
            broad |= ((bits >> BIT[name]) & 1).astype(bool)

        codeplane = np.zeros((CELL_PX, CELL_PX), dtype=np.uint8)
        codeplane[standalone] = 1
        codeplane[standalone & broad] = 4
        codeplane[standalone & precise] = 3      # precise beats broad where they overlap
        codeplane[fn] = 2                        # FP and FN are disjoint

        for r in rows:
            vh, vw = r['valid_h'], r['valid_w']
            tile = np.zeros((512, 512), dtype=np.uint8)
            tile[:vh, :vw] = codeplane[r['row_off']:r['row_off'] + vh,
                                       r['col_off']:r['col_off'] + vw]
            Image.fromarray(tile).save(CODE_ROOT / r['name'])
        for c in range(5):
            stats[c] += int((codeplane == c).sum())
        if n % 10 == 0 or n == len(cells):
            print(f'  {n}/{len(cells)} training cells')

    del model; torch.cuda.empty_cache()
    tot = sum(stats.values())
    names = {0: 'ordinary (incl. boundary FP)', 1: 'standalone FP, unattributed',
             2: 'false negative', 3: 'standalone FP, precise layer',
             4: 'standalone FP, broad layer'}
    print('\nweight-code composition over the training half')
    for c in range(5):
        print(f'  {c}  {names[c]:<32} {stats[c]*PIXEL_M**2/1e6:8.4f} km²  {100*stats[c]/tot:6.2f}%')
    pd.DataFrame([{'code': c, 'meaning': names[c], 'km2': stats[c]*PIXEL_M**2/1e6,
                   'pct': 100*stats[c]/tot} for c in range(5)]).to_csv(RUN / 'weight_codes.csv', index=False)


have_codes = CODE_ROOT.exists() and len(list(CODE_ROOT.glob('*.png'))) == 2256
if have_codes and not FORCE_REBUILD_CODES:
    print('weight codes already present')
else:
    build_weight_codes()
assert len(list(CODE_ROOT.glob('*.png'))) == 2256

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/339M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/339M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b5-finetuned-cityscapes-1024-1024 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([19, 768, 1, 1]) in the checkpoint and torch.Size([2, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([19]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  checkpoint verified: 1172 tensors loaded exactly from /content/drive/MyDrive/Parking_finetuning_run/best_model.ckpt


preprocessor_config.json:   0%|          | 0.00/273 [00:00<?, ?B/s]

  10/50 training cells
  20/50 training cells
  30/50 training cells
  40/50 training cells
  50/50 training cells

weight-code composition over the training half
  0  ordinary (incl. boundary FP)      49.0990 km²   98.20%
  1  standalone FP, unattributed        0.1195 km²    0.24%
  2  false negative                     0.1624 km²    0.32%
  3  standalone FP, precise layer       0.4222 km²    0.84%
  4  standalone FP, broad layer         0.1968 km²    0.39%


## 8. Targeted fine-tuning

Weighted cross-entropy, reproducing Hugging Face's own loss path (logits bilinearly upsampled
to label size, `ignore_index=255`) but with `reduction='none'` so the weight map can be applied.

The loss is normalised by the **sum of weights**, not by the pixel count. Dividing by count
would make the weighted loss numerically larger than the generic run's and effectively raise
the learning rate, which would confound the comparison with the very thing being tested.

Optimiser, LR, batch size, epochs, seed, the cell-level fit/validation split and the
best-epoch-on-validation-IoU rule are all identical to the generic run.

In [9]:
import random, time
from modeling import BASE_MODEL, PATCH_SIZE

WEIGHTS = torch.tensor([W_ORDINARY, W_FP_OTHER, W_FN, W_FP_PRECISE, W_FP_BROAD],
                       dtype=torch.float32, device=device)
print('code weights:', WEIGHTS.tolist())


class CodedPatchDataset(torch.utils.data.Dataset):
    """SourcePatchDataset plus the per-pixel weight code for the same patch."""
    def __init__(self, rows, patch_root, code_root, processor):
        self.base = SourcePatchDataset(rows, patch_root, str(TIF_ROOT), processor,
                                       ignore_padding=True)
        self.rows, self.code_root = rows, Path(code_root)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        enc = self.base[i]
        with Image.open(self.code_root / self.rows[i]['name']) as im:
            enc['weight_code'] = torch.from_numpy(np.array(im).astype(np.int64))
        return enc


def weighted_loss(logits, labels, codes):
    up = F.interpolate(logits, size=labels.shape[-2:], mode='bilinear', align_corners=False)
    valid = labels != 255
    safe = labels.clone(); safe[~valid] = 0
    ce = F.cross_entropy(up.float(), safe.long(), reduction='none')
    w = WEIGHTS[codes] * valid
    return (ce * w).sum() / w.sum().clamp(min=1.0)


@torch.no_grad()
def val_scores(model, loader):
    model.eval(); tp = fp = fn = 0
    for batch in loader:
        gt = batch['labels'].to(device)
        logits = model(pixel_values=batch['pixel_values'].to(device))[0]
        up = F.interpolate(logits, size=gt.shape[-2:], mode='bilinear', align_corners=False)
        pr = up.argmax(1); ok = gt != 255
        tp += int(((pr == 1) & (gt == 1) & ok).sum())
        fp += int(((pr == 1) & (gt == 0) & ok).sum())
        fn += int(((pr != 1) & (gt == 1) & ok).sum())
    d = lambda a, b: a / b if b else 0.0
    return d(tp, tp + fp), d(tp, tp + fn), d(tp, tp + fp + fn)


def train_targeted():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    rng = np.random.default_rng(SEED)

    tr = IDX[(IDX['split'] == 'train') & IDX['kept']]
    cells = np.array(sorted(tr['cell'].unique())); rng.shuffle(cells)
    n_val = max(1, int(round(0.2 * len(cells))))
    val_cells, fit_cells = set(cells[:n_val]), set(cells[n_val:])
    fit_rows = tr[tr['cell'].isin(fit_cells)].to_dict('records')
    val_rows = tr[tr['cell'].isin(val_cells)].to_dict('records')
    print(f'fit {len(fit_cells)} cells / {len(fit_rows)} patches | '
          f'val {len(val_cells)} cells / {len(val_rows)} patches')
    pd.DataFrame([{'cell': c, 'role': 'fit' if c in fit_cells else 'validation'}
                  for c in sorted(fit_cells | val_cells)]).to_csv(RUN / 'fit_val_split.csv', index=False)

    proc = make_processor()
    root = str(FT / 'patches' / 'train')
    fit_ds = CodedPatchDataset(fit_rows, root, CODE_ROOT, proc)
    val_ds = SourcePatchDataset(val_rows, root, str(TIF_ROOT), proc, ignore_padding=True)
    fit_dl = DataLoader(fit_ds, batch_sampler=TileBatchSampler(fit_rows, TRAIN_BATCH, SEED, True),
                        num_workers=NUM_WORKERS, pin_memory=True)
    val_dl = DataLoader(val_ds, batch_size=EVAL_BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

    model = load_checkpoint(make_model(), str(ZERO_SHOT)).to(device)
    p0, r0, i0 = val_scores(model, val_dl)
    print(f'zero-shot on validation cells: P {p0:.4f}  R {r0:.4f}  IoU {i0:.4f}')

    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR, eps=1e-8)
    scaler = torch.cuda.amp.GradScaler(enabled=True)
    hist = [{'epoch': 0, 'train_loss': None, 'val_precision': round(p0, 4),
             'val_recall': round(r0, 4), 'val_parking_iou': round(i0, 4), 'note': 'zero-shot'}]
    best, best_epoch = -1.0, None

    for ep in range(1, EPOCHS + 1):
        model.train(); losses = []; t0 = time.time()
        for step, batch in enumerate(fit_dl, 1):
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=True):
                logits = model(pixel_values=batch['pixel_values'].to(device, non_blocking=True))[0]
            loss = weighted_loss(logits, batch['labels'].to(device, non_blocking=True),
                                 batch['weight_code'].to(device, non_blocking=True))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            losses.append(float(loss.detach()))
            if step % 100 == 0:
                print(f'  ep {ep} step {step}/{len(fit_dl)} loss {np.mean(losses[-100:]):.4f}')
        p, r, i = val_scores(model, val_dl)
        print(f'epoch {ep}: loss {np.mean(losses):.4f} | val P {p:.4f} R {r:.4f} IoU {i:.4f}'
              f' | {time.time()-t0:.0f}s' + ('  <- best' if i > best else ''))
        hist.append({'epoch': ep, 'train_loss': round(float(np.mean(losses)), 4),
                     'val_precision': round(p, 4), 'val_recall': round(r, 4),
                     'val_parking_iou': round(i, 4), 'note': ''})
        if i > best:
            best, best_epoch = i, ep
            torch.save({'state_dict': model.state_dict(), 'epoch': ep, 'val_parking_iou': i,
                        'zero_shot_val_parking_iou': i0, 'base_model': BASE_MODEL,
                        'processor_size': PATCH_SIZE, 'seed': SEED,
                        'weights': WEIGHTS.tolist()}, TARGETED_CKPT)

    pd.DataFrame(hist).to_csv(RUN / 'targeted_log.csv', index=False)
    print(f'\nbest epoch {best_epoch}: validation parking IoU {i0:.4f} -> {best:.4f}')
    del model; torch.cuda.empty_cache()


if TARGETED_CKPT.exists() and (RUN / 'targeted_log.csv').exists() and not FORCE_RETRAIN:
    print('targeted checkpoint already present; set FORCE_RETRAIN=True to redo')
    display(pd.read_csv(RUN / 'targeted_log.csv'))
else:
    train_targeted()

code weights: [1.0, 1.0, 3.0, 5.0, 2.0]
targeted checkpoint already present; set FORCE_RETRAIN=True to redo


,epoch,train_loss,val_precision,val_recall,val_parking_iou,note
0,0,NaN,0.5670,0.9043,0.5349,zero-shot
1,1,0.0982,0.8711,0.5323,0.4935,NaN
2,2,0.0711,0.8316,0.7103,0.6210,NaN
3,3,0.0531,0.8199,0.7312,0.6300,NaN
4,4,0.0382,0.7875,0.7581,0.6294,NaN
5,5,0.0307,0.7502,0.7944,0.6282,NaN
6,6,0.0259,0.7401,0.7907,0.6189,NaN


## 9. Three-arm, category-resolved evaluation

Same 50 held-out cells, same preprocessing, for every arm. Each complete 1 km cell is
reassembled before any distance is measured, so patch seams are never mistaken for car-park
boundaries. FP is split into dilation (≤ 5 m of a label) and standalone, and the standalone
part is attributed by the peeling order of §4.2. FN is split against the *prediction*, not
against the reference edge.

**Sanity check built in:** arm A must reproduce the first experiment's held-out zero-shot row,
micro P 0.5190 / R 0.8819 / IoU 0.4853. If it does not, something upstream has drifted and the
comparison is void.

In [10]:
TEST = IDX[(IDX['split'] == 'test') & IDX['kept']].sort_values(['cell', 'row_off', 'col_off'])
counts = TEST.groupby('cell').size()
assert len(counts) == 50 and (counts == 64).all(), 'held-out set incomplete'
TEST_ROWS = TEST.to_dict('records')
print(f'held-out: {len(TEST_ROWS)} patches / {len(counts)} cells')

BANDS_M = (2.0, 5.0, 10.0)


def score_arm(ckpt, name):
    model = load_checkpoint(make_model(), str(ckpt)).to(device).eval()
    per_cell, cat, bands = {}, {k: 0 for k in PEEL + ['other']}, \
        {m: dict(fp_dil=0, fp_std=0, fn_ero=0, fn_std=0) for m in BANDS_M}

    cells = sorted({r['cell'] for r in TEST_ROWS})
    for n, cell in enumerate(cells, 1):
        rows = [r for r in TEST_ROWS if r['cell'] == cell]
        preds = predict_rows(model, rows, str(FT / 'patches' / 'test'))
        masks = []
        for r in rows:
            with Image.open(FT / 'patches' / 'test' / 'Masks' / r['name']) as im:
                masks.append(np.array(im) == 1)
        ref  = assemble(list(zip(rows, masks)))
        pred = assemble(list(zip(rows, preds)))

        tp_m, fp_m, fn_m = pred & ref, pred & ~ref, ~pred & ref
        per_cell[cell] = [int(tp_m.sum()), int(fp_m.sum()), int(fn_m.sum())]

        d_ref  = ndimage.distance_transform_edt(~ref)  if ref.any()  else None
        d_pred = ndimage.distance_transform_edt(~pred) if pred.any() else None
        for m in BANDS_M:
            px = m / PIXEL_M
            if d_ref is not None:
                near = d_ref <= px
                bands[m]['fp_dil'] += int((fp_m & near).sum())
                bands[m]['fp_std'] += int((fp_m & ~near).sum())
            else:
                bands[m]['fp_std'] += int(fp_m.sum())
            if d_pred is not None:
                near = d_pred <= px
                bands[m]['fn_ero'] += int((fn_m & near).sum())
                bands[m]['fn_std'] += int((fn_m & ~near).sum())
            else:
                bands[m]['fn_std'] += int(fn_m.sum())

        # attribute the 5 m standalone FP, most specific layer first
        rem = fp_m & (d_ref > DIL_PX) if d_ref is not None else fp_m.copy()
        bits = CATEGORY[cell]
        for layer in PEEL:
            hit = rem & ((bits >> BIT[layer]) & 1).astype(bool)
            cat[layer] += int(hit.sum()); rem &= ~hit
        cat['other'] += int(rem.sum())

        del d_ref, d_pred
        if n % 10 == 0 or n == len(cells):
            print(f'  [{name}] {n}/{len(cells)} cells')

    del model; torch.cuda.empty_cache()
    return per_cell, cat, bands


def summarise(name, per_cell, cat, bands):
    a = PIXEL_M ** 2 / 1e6
    tp = sum(v[0] for v in per_cell.values())
    fp = sum(v[1] for v in per_cell.values())
    fn = sum(v[2] for v in per_cell.values())
    f = lambda x, y: x / y if y else float('nan')
    rows = [{'model': name, 'aggregation': 'micro',
             'precision': round(f(tp, tp+fp), 4), 'recall': round(f(tp, tp+fn), 4),
             'iou': round(f(tp, tp+fp+fn), 4), 'tp_km2': round(tp*a, 4),
             'fp_km2': round(fp*a, 4), 'fn_km2': round(fn*a, 4),
             'predicted_km2': round((tp+fp)*a, 4), 'reference_km2': round((tp+fn)*a, 4)}]
    pc = [(f(v[0], v[0]+v[1]), f(v[0], v[0]+v[2]), f(v[0], sum(v))) for v in per_cell.values() if sum(v)]
    rows.append({'model': name, 'aggregation': 'macro',
                 'precision': round(float(np.nanmean([x[0] for x in pc])), 4),
                 'recall':    round(float(np.nanmean([x[1] for x in pc])), 4),
                 'iou':       round(float(np.nanmean([x[2] for x in pc])), 4)})
    band_rows = [{'model': name, 'distance_m': m,
                  'fp_dilation_km2': round(b['fp_dil']*a, 4),
                  'fp_standalone_km2': round(b['fp_std']*a, 4),
                  'fn_erosion_km2': round(b['fn_ero']*a, 4),
                  'fn_standalone_km2': round(b['fn_std']*a, 4)} for m, b in bands.items()]
    cat_rows = [{'model': name, 'category': k, 'standalone_fp_km2': round(v*a, 4)}
                for k, v in cat.items()]
    return rows, band_rows, cat_rows


ARMS = [('A zero-shot', ZERO_SHOT)]
if HAVE_GENERIC:
    ARMS.append(('B generic FT', GENERIC_CKPT))
ARMS.append(('C targeted FT', TARGETED_CKPT))

all_rows, all_bands, all_cats, RAW = [], [], [], {}
for name, ckpt in ARMS:
    print(f'\n=== scoring {name} ===')
    pc, cat, bands = score_arm(ckpt, name)
    RAW[name] = (pc, cat, bands)
    r, b, c = summarise(name, pc, cat, bands)
    all_rows += r; all_bands += b; all_cats += c

EVAL  = pd.DataFrame(all_rows)
BANDS = pd.DataFrame(all_bands)
CATS  = pd.DataFrame(all_cats)
EVAL.to_csv(RUN / 'evaluation_3arm.csv', index=False)
BANDS.to_csv(RUN / 'boundary_bands_3arm.csv', index=False)
CATS.to_csv(RUN / 'standalone_fp_by_category.csv', index=False)

z = EVAL[(EVAL.model == 'A zero-shot') & (EVAL.aggregation == 'micro')].iloc[0]
print(f'\nsanity check - arm A micro: P {z.precision} R {z.recall} IoU {z.iou}')
print('expected from the first experiment: P 0.5190  R 0.8819  IoU 0.4853')
if not (abs(z.precision-0.5190) < 0.002 and abs(z.recall-0.8819) < 0.002):
    print('*** MISMATCH - the pipeline has drifted; do not report the comparison ***')
else:
    print('reproduced - the two experiments share a baseline')

held-out: 3200 patches / 50 cells

=== scoring A zero-shot ===


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b5-finetuned-cityscapes-1024-1024 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([19, 768, 1, 1]) in the checkpoint and torch.Size([2, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([19]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  checkpoint verified: 1172 tensors loaded exactly from /content/drive/MyDrive/Parking_finetuning_run/best_model.ckpt
  [A zero-shot] 10/50 cells
  [A zero-shot] 20/50 cells
  [A zero-shot] 30/50 cells
  [A zero-shot] 40/50 cells
  [A zero-shot] 50/50 cells

=== scoring B generic FT ===


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b5-finetuned-cityscapes-1024-1024 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([19, 768, 1, 1]) in the checkpoint and torch.Size([2, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([19]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  checkpoint verified: 1172 tensors loaded exactly from /content/drive/MyDrive/Parking_finetuning_run/finetuned.ckpt
  [B generic FT] 10/50 cells
  [B generic FT] 20/50 cells
  [B generic FT] 30/50 cells
  [B generic FT] 40/50 cells
  [B generic FT] 50/50 cells

=== scoring C targeted FT ===


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/segformer-b5-finetuned-cityscapes-1024-1024 and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([19, 768, 1, 1]) in the checkpoint and torch.Size([2, 768, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([19]) in the checkpoint and torch.Size([2]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  checkpoint verified: 1172 tensors loaded exactly from /content/drive/MyDrive/Parking_targeted_run/targeted.ckpt
  [C targeted FT] 10/50 cells
  [C targeted FT] 20/50 cells
  [C targeted FT] 30/50 cells
  [C targeted FT] 40/50 cells
  [C targeted FT] 50/50 cells

sanity check - arm A micro: P 0.519 R 0.8819 IoU 0.4853
expected from the first experiment: P 0.5190  R 0.8819  IoU 0.4853
reproduced - the two experiments share a baseline


## 10. The tables this notebook exists to produce

**Table 1** overall accuracy per arm — does targeting keep the recall the generic run lost?

**Table 2** standalone FP by category, and the **removal rate** relative to zero-shot. This is
the answer to the question the generic experiment could not settle. Read it like this:

* removal rates roughly **flat** across categories, in both arms → both models simply
  contracted, and no claim about learning specific confusions is supportable;
* arm C removes **more** in the targeted categories (road, curtilage, OSM parking, sports) than
  arm B does, while holding recall → targeting worked, and the typology earned its keep;
* arm C removes more everywhere including `other` → still a contraction, just a harder one.

In [11]:
pd.set_option('display.width', 200)

print('=== Table 1  overall accuracy on the 50 held-out cells ===')
t1 = EVAL[EVAL.aggregation == 'micro'][
    ['model', 'precision', 'recall', 'iou', 'tp_km2', 'fp_km2', 'fn_km2',
     'predicted_km2', 'reference_km2']].reset_index(drop=True)
display(t1)
display(EVAL[EVAL.aggregation == 'macro'][['model', 'precision', 'recall', 'iou']]
        .reset_index(drop=True))

print('\n=== Table 2  standalone FP (>5 m) by category, and removal vs zero-shot ===')
piv = CATS.pivot(index='category', columns='model', values='standalone_fp_km2')
base = piv['A zero-shot']
t2 = pd.DataFrame({'zero_shot_km2': base})
for name, _ in ARMS[1:]:
    t2[f'{name}_km2'] = piv[name]
    t2[f'{name}_removed_%'] = (100 * (base - piv[name]) / base.replace(0, np.nan)).round(1)
order = [c for c in PEEL + ['other'] if c in t2.index]
t2 = t2.loc[order].round(4)
t2.loc['TOTAL'] = t2.loc[[c for c in order]].sum(numeric_only=True).round(4)
for name, _ in ARMS[1:]:
    tot_b, tot_f = base.sum(), piv[name].sum()
    t2.loc['TOTAL', f'{name}_removed_%'] = round(100 * (tot_b - tot_f) / tot_b, 1)
display(t2)

print('\n=== Table 3  boundary bands ===')
display(BANDS[BANDS.distance_m == 5.0].reset_index(drop=True))

print('\n=== Selectivity read-out ===')
TARGETED_CATS = ['road_wide', 'curtilage', 'osm_parking', 'sports']
for name, _ in ARMS[1:]:
    col = f'{name}_removed_%'
    tgt = t2.loc[[c for c in TARGETED_CATS if c in t2.index], col].mean()
    oth = t2.loc[[c for c in order if c not in TARGETED_CATS], col].mean()
    print(f'{name}: targeted categories removed {tgt:.1f}% on average, '
          f'others {oth:.1f}%  ->  selectivity gap {tgt-oth:+.1f} points')

rec = EVAL[EVAL.aggregation == 'micro'].set_index('model')['recall']
print('\nrecall by arm:', {k: float(v) for k, v in rec.items()})

t1.to_csv(RUN / 'table1_overall.csv', index=False)
t2.to_csv(RUN / 'table2_category_removal.csv')
print(f'\nwrote all outputs to {RUN}')

=== Table 1  overall accuracy on the 50 held-out cells ===


,model,precision,recall,iou,tp_km2,fp_km2,fn_km2,predicted_km2,reference_km2
0,A zero-shot,0.5190,0.8819,0.4853,1.4322,1.3274,0.1917,2.7596,1.6239
1,B generic FT,0.7664,0.7548,0.6136,1.2257,0.3736,0.3982,1.5993,1.6239
2,C targeted FT,0.7791,0.6935,0.5795,1.1262,0.3194,0.4978,1.4456,1.6239


,model,precision,recall,iou
0,A zero-shot,0.4773,0.8777,0.4473
1,B generic FT,0.7553,0.7205,0.5833
2,C targeted FT,0.7625,0.6546,0.5442



=== Table 2  standalone FP (>5 m) by category, and removal vs zero-shot ===


,zero_shot_km2,B generic FT_km2,B generic FT_removed_%,C targeted FT_km2,C targeted FT_removed_%
category,,,,,
building,0.0762,0.0172,77.4,0.0171,77.6
osm_parking,0.0578,0.0331,42.7,0.0273,52.8
sports,0.0241,0.0024,90.0,0.0014,94.2
road_wide,0.2164,0.0402,81.4,0.0247,88.6
curtilage,0.1947,0.0472,75.8,0.0293,85.0
brownfield,0.0175,0.0141,19.4,0.0137,21.7
industrial,0.2363,0.0664,71.9,0.0528,77.7
other,0.1307,0.0229,82.5,0.0272,79.2
TOTAL,0.9537,0.2435,74.5,0.1935,79.7



=== Table 3  boundary bands ===


,model,distance_m,fp_dilation_km2,fp_standalone_km2,fn_erosion_km2,fn_standalone_km2
0,A zero-shot,5.0,0.3737,0.9537,0.0858,0.1059
1,B generic FT,5.0,0.1302,0.2434,0.1684,0.2298
2,C targeted FT,5.0,0.1259,0.1935,0.1617,0.3361



=== Selectivity read-out ===
B generic FT: targeted categories removed 72.5% on average, others 62.8%  ->  selectivity gap +9.7 points
C targeted FT: targeted categories removed 80.1% on average, others 64.0%  ->  selectivity gap +16.1 points

recall by arm: {'A zero-shot': 0.8819, 'B generic FT': 0.7548, 'C targeted FT': 0.6935}

wrote all outputs to /content/drive/MyDrive/Parking_targeted_run


## 11. How to read this against the dissertation

Whatever comes out, §4.8 gains a sentence it can defend. Three outcomes, three sentences:

1. **Selectivity gap large and positive, recall held.** The typology did more than describe the
   error — it improved the intervention. §4.8 can say local supervision corrects definitional
   disagreement *specifically*, and the decomposition is what made that possible.
2. **Selectivity gap near zero in both arms.** Both models contracted; the standalone-FP fall in
   the generic run was not category-specific after all. §4.8 must then state that the
   definitional reading is unsupported — which is a stronger result than the hedge currently
   in the write-up, because it is a measurement rather than a caveat.
3. **Gap positive but recall still falls.** Targeting reaches the right pixels but the model
   still shrinks. That points at the decision rule rather than the training signal, and makes
   the threshold-calibration comparison the obvious next step.

None of these requires the chapter's headline figures to change. Arm A is the same released
model on the same held-out cells, and the sanity check in section 9 is what keeps it honest.